требуется реализовать функцию батч-нормализации без использования стандартной функции со следующими упрощениями:

    Параметр Бета принимается равным 0.
    Параметр Гамма принимается равным 1.
    Функция должна корректно работать только на этапе обучения.
    Вход имеет размерность число элементов в батче * длина каждого инстанса.


In [1]:
import numpy as np
import torch
import torch.nn as nn

def custom_batch_norm1d(input_tensor, eps):
    batch_mean = input_tensor.mean(dim=0, keepdim=True)
    batch_var = input_tensor.var(dim=0, unbiased=False, keepdim=True)
    normed_tensor = (input_tensor - batch_mean) / torch.sqrt(batch_var + eps)
    return normed_tensor


input_tensor = torch.Tensor([[0.0, 0, 1, 0, 2], [0, 1, 1, 0, 10]])
batch_norm = nn.BatchNorm1d(input_tensor.shape[1], affine=False)



In [2]:
# Проверка происходит автоматически вызовом следующего кода

import numpy as np
all_correct = True
for eps_power in range(10):
    eps = np.power(10., -eps_power)
    batch_norm.eps = eps
    batch_norm_out = batch_norm(input_tensor)
    custom_batch_norm_out = custom_batch_norm1d(input_tensor, eps)

    all_correct &= torch.allclose(batch_norm_out, custom_batch_norm_out)
    all_correct &= batch_norm_out.shape == custom_batch_norm_out.shape
print(all_correct)

True


### Кратко о ноутбуке

Реализована **упрощённая пакетная нормализация для 2D-входа** `[N, C]` (число объектов в батче × длина признакового вектора): для каждого признака вычисляются среднее и дисперсия **только по размерности батча**, затем применяется формула \((x - \mu) / \sqrt{\sigma^2 + \varepsilon}\). Параметры **γ = 1** и **β = 0** соответствуют отсутствию аффинного преобразования после нормирования (как у `nn.BatchNorm1d(..., affine=False)`).

**Проверка:** для одного и того же `input_tensor` выход `custom_batch_norm1d` сравнивается с `nn.BatchNorm1d` без обучаемых весов при разных значениях `eps` (степени 10⁻ᵏ). Совпадение по `torch.allclose` и форме тензора подтверждает корректность реализации в **режиме обучения** (как в задании).